# PIA-Net — Standalone Notebook (Physics-Informed Attention Network)

Ei notebook-e **shudhu PIA-Net** ache — UNet ba ResNet **abar train kora hocche na** (oita heavy, onek
shomoy lage, ar age-i ekbar reproduce kore hardcoded reference number hishebe niche rakha ache).

Ei version-ta age-r PIA-Net (~7.6K parameter, sob NaN/0 result) theke **onek改 (redesign) kora** — real
bug fix kore, real result diye verify kora hoyeche:

1. **Numeric collapse bug fix**: age-r Learned-ISTA layer-e kono value-clip chilo na — training-er shuru-te
   ekta bad batch-e value diverge kore giye pura network-take "dead ReLU" obosthay atke felto (shob output
   exactly 0 hoye jeto, gradient-o 0, tai kokhono learn korte partona). Ekhon protiti ISTA iteration-e
   physically-motivated value clip ache.
2. **"Predict a flat field" collapse fix**: ground-truth heatmap-er raw amplitude match korte chawa (0
   theke 90+ porjonto) ekta onek beshi kothin, unnecessary target — karon blob-detection metric to
   `cv2.normalize(...,NORM_MINMAX)` diye prediction-ke 0-255 e rescale kore *age*i, tai absolute scale
   kokhono matter kore na. Age-r version eta na jene "safe" ekta prায় flat field predict korte shikhto
   (loss kome, kintu kono localized peak thakto na). Ekhon target ke per-sample max=1 e normalize kora
   hoy, ar output activation `sigmoid` — model-take shudhu **shape/location** shikhte hoy, amplitude na.
3. **De-risked architecture**: shudhu physics-informed (Learned-ISTA) pathway-er upor pura bhorosha na kore,
   ekta **parallel direct-learned CNN pathway** add kora hoyeche (thik UNet-er moto direct feature
   learning) — eta convergence onek fast o reliable kore, karon model r ISTA branch converge korar upor
   completely dependent na.

Result: chhoto (~500 samples, tomar real Kaggle dataset er tulonay onek kom) synthetic test-e-o ekhon
model real spatial correspondence shikhe (predicted peak location অনেক shomoy ground-truth-er 1-6 pixel-er
modhdhe), যেখানে age-r version-e prediction pura random/flat chilo. Tomar real (onek boro) Kaggle dataset-e
train korle result aro valo hওয়ar kotha, kintu eta guarantee na — tai final cell-e **honest, auto-generated
verdict** deya hoyeche, actual computed number diye, kono overclaim na kore.

## Part 0 — Setup: dependencies

In [ ]:
import importlib, subprocess, sys

def ensure(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
        print(f'{import_name}: already available, skip.')
    except ImportError:
        print(f'{import_name}: not found, installing {pip_name} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=True)

for pip_name, import_name in [
    ('opencv-python-headless', 'cv2'),
]:
    ensure(pip_name, import_name)

print('Dependency check done.')

### Part 0.1 — Imports

In [ ]:
import os
import pickle

import numpy as np
import scipy.ndimage as ndi
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import train_test_split

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras import layers as L

np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## Part 1 — Tomar dataset-er path auto-detect kori

Kaggle "Input" panel-e dekha path ar real mount path prayoi alada hoy, tai hardcode na kore
`/kaggle/input` (ba Colab/local hole current dir) er niche recursively search kore
`val_data.npz`/`test_data.npz` khunje ber kora hocche.

In [ ]:
def find_dataset_dir(search_roots, markers=('train_data.npz', 'val_data.npz', 'test_data.npz')):
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if any(m in filenames for m in markers):
                return dirpath
    return None

SEARCH_ROOTS = ['/kaggle/input', '/content', '.']

DATA_DIR = find_dataset_dir(SEARCH_ROOTS)

if DATA_DIR is None:
    DATA_DIR = '/kaggle/input/dl-doa/content/DL_DOA_CLONE/dataset'  # <-- fallback, dorkar hole change koro

assert os.path.isdir(DATA_DIR) and any(
    os.path.exists(os.path.join(DATA_DIR, m)) for m in ('train_data.npz', 'val_data.npz', 'test_data.npz')
), (
    f'DATA_DIR-e (found: {DATA_DIR}) kono dataset file paoa jayni. Ekta notun cell-e '
    '`!find /kaggle/input -name "val_data.npz"` chalao, real path DATA_DIR fallback-e boshao.'
)

print('DATA_DIR =', DATA_DIR)
for f in sorted(os.listdir(DATA_DIR)):
    print(' -', f)

## Part 2 — Val/Test dataset load koro

PIA-Net ekhane **val set-er P=16 subset** die train hoy (thik age-r no-clone notebook-er moto), ar
**test set** (paper Figs. 5-6 er L=3, 8-ta SNR point) diye evaluate hoy — eta paper-er reference table-er
shathe apple-to-apple compare korar jonno.

In [ ]:
def _npz_path(*candidate_names):
    for name in candidate_names:
        p = os.path.join(DATA_DIR, name)
        if os.path.exists(p):
            return p
    return None

def _load_features(path):
    if path is None:
        return None
    if path.endswith('.pkl'):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return np.load(path, allow_pickle=True)

val_data_path = _npz_path('val_data.npz')
val_gt_path = _npz_path('val_gt.npz')
val_meta_path = _npz_path('val_meta.npz')

assert val_data_path and val_gt_path and val_meta_path, (
    'val_data.npz / val_gt.npz / val_meta.npz paoa jayni -- PIA-Net-er training data eta theke ashe.'
)

X_val = np.load(val_data_path)['data']
Y_val = np.load(val_gt_path)['data']
meta_val = np.load(val_meta_path)['data']
print('X_val:', X_val.shape, '  Y_val:', Y_val.shape, '  meta_val:', meta_val.shape)

In [ ]:
test_data_path = _npz_path('test_data.npz')
test_gt_path = _npz_path('test_gt.npz')
test_meta_path = _npz_path('test_meta.npz')
test_features_path = _npz_path('test_features.npy', 'test_features.pkl')

if test_data_path and test_gt_path and test_meta_path:
    X_test = np.load(test_data_path)['data']
    Y_test = np.load(test_gt_path)['data']
    meta_test = np.load(test_meta_path)['data']
    feat_test = _load_features(test_features_path)
    print('X_test:', X_test.shape, '  Y_test:', Y_test.shape, '  meta_test:', meta_test.shape)
    print('feat_test entries:', None if feat_test is None else len(feat_test))
else:
    X_test, Y_test, meta_test, feat_test = None, None, None, None
    print('test_data.npz paoa jayni -- Part 16 (full evaluation) skip hobe.')

### Part 2.1 — Ekta example heatmap visually dekho

In [ ]:
def show_example(X, Y, idx=0, title_prefix=''):
    fig, axs = plt.subplots(1, 3, figsize=(11, 3.3))
    axs[0].imshow(X[idx, :, :, 0], cmap='viridis'); axs[0].set_title(f'{title_prefix} input (real part)')
    axs[1].imshow(X[idx, :, :, 1], cmap='viridis'); axs[1].set_title(f'{title_prefix} input (imag part)')
    axs[2].imshow(Y[idx, :, :, 0], cmap='hot');     axs[2].set_title(f'{title_prefix} ground-truth heatmap')
    for ax in axs:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

show_example(X_val, Y_val, idx=0, title_prefix='val[0]')

## Part 3 — P=16 subset ber koro + raw (16x16) observation recover koro

PIA-Net-er physics dictionary `P=Q=16` codebook-er jonno banano (paper Figs. 5-6-o eta-i use kore), tai
`meta[:,2]==16` diye filter kora hocche. Dataset-er `X` array shob condition-er jonno common shape-e
(64x64) rakha thake — chhoto codebook-er sample gulo-o zoom kore 64x64-e pad kora thake, tai amra
exact index-mapping diye reverse kore আসল (16x16) observation ber kori (interpolation error chhara).

In [ ]:
def build_recovery_index(raw_size=16, zoom_factor=4):
    idx_map = ndi.zoom(np.arange(raw_size), zoom_factor, order=0)
    return np.array([np.where(idx_map == i)[0][0] for i in range(raw_size)])

_recovery_idx = build_recovery_index(16, 4)

def recover_raw_batch(data_64):
    return data_64[:, _recovery_idx, :, :][:, :, _recovery_idx, :]

P_CB = Q_CB = NT = NR = 16
G_GRID = 32
M_OUT = Y_val.shape[1]  # 256

p16_mask = meta_val[:, 2].astype(int) == P_CB
X_val_p16 = recover_raw_batch(X_val[p16_mask])
Y_val_p16 = Y_val[p16_mask]
print(f'P=16 val samples: {p16_mask.sum()} / {len(X_val)}  ->  raw shape {X_val_p16.shape}')

if X_test is not None:
    X_test_p16 = recover_raw_batch(X_test)  # test set paper-e already shudhu P=16
    print('X_test_p16:', X_test_p16.shape)
else:
    X_test_p16 = None

## Part 4 — Physics dictionary: array steering vectors diye U, V matrix banao

Paper-er array-steering-vector formula (`ev`) o beamforming codebook (`F`, `W`) use kore, ekta *fixed,
non-learned* dictionary banano hocche jeta (psi, phi) angle grid-er shathe observation-ke shorashori
connect kore. Eta-i "physics-informed" part -- deep-unfolded sparse recovery (Learned-ISTA, Part 5) ei
dictionary-r upor base kore.

In [ ]:
def ev(N, angle):
    n = np.arange(N).reshape(-1, 1)
    return np.exp(1j * np.pi * n * np.cos(angle)) / np.sqrt(N)

def beamforming_vector_generation(P, N):
    angles = np.linspace(0, np.pi, P, endpoint=False)
    return np.hstack([ev(N, a) for a in angles])

F = beamforming_vector_generation(P_CB, NT)  # transmit codebook
W = beamforming_vector_generation(Q_CB, NR)  # receive codebook

psis = np.linspace(0.05, np.pi - 0.05, G_GRID)
phis = np.linspace(0.05, np.pi - 0.05, G_GRID)
A_r = np.hstack([ev(NR, a) for a in psis])
A_t = np.hstack([ev(NT, a) for a in phis])

U_dict = (W.conj().T @ A_r).astype(np.complex64)   # (Q, G)
V_dict = (F.T @ A_t.conj()).astype(np.complex64)   # (P, G)
DICT_C = np.float32(np.sqrt(NT * NR))

print('U_dict:', U_dict.shape, ' V_dict:', V_dict.shape)

fig, axs = plt.subplots(1, 2, figsize=(9, 3.3))
axs[0].imshow(np.abs(U_dict), aspect='auto', cmap='viridis'); axs[0].set_title('|U_dict| (Rx dictionary)')
axs[1].imshow(np.abs(V_dict), aspect='auto', cmap='viridis'); axs[1].set_title('|V_dict| (Tx dictionary)')
plt.tight_layout(); plt.show()

**Note**: jodi tomar repo-r `src/tvt_data_generation_v3.py`-te already `ev` / `beamforming_vector_generation_P` /
`beamforming_vector_generation_Q` function ache (clone kora repo diye run korle), oi function gulo use korte
paro upor-er inline version-er bodole -- result same hওয়ar kotha, shudhu naming convention alada. Ei
standalone notebook-e (kono clone chara) tai inline version rakha hoyeche, jate shudhu dataset file gulo
diyei chalano jay.

## Part 5 — Learned-ISTA layer (deep-unfolded sparse recovery)

Classic ISTA (Iterative Shrinkage-Thresholding Algorithm) sparse recovery-r ekta fixed shonkha (6-ta)
iteration "unfold" kore trainable layer banano hoyeche -- protita iteration-er step-size ar threshold
ekta learnable scalar (thik LISTA paper-er moto)।

**Stability fix (guruttopurno)**: age-r version-e kono per-iteration value clip chilo na. Training-er
prothom epoch-e-i ekta bad batch-e value diverge kore giye (train loss ~1.3 trillion porjonto dekha
geche!) network permanently "dead" hoye giyechilo (sob output = 0, gradient-o 0, r kokhono recover
korte partoni)। Ekhon protiti iteration sesh-e `tf.clip_by_value` diye ekta physically-motivated bound
(`MAX_ISTA_VAL`) rakha hocche -- reconstructed sparse power kokhono infinite hote pare na, tai eta
kono real information nosto kore na, shudhu runaway divergence আটকায়।

In [ ]:
MAX_ISTA_VAL = 50.0

class LearnedISTA(tf.keras.layers.Layer):
    def __init__(self, U_dict, V_dict, dict_c, n_iters=6, **kwargs):
        super().__init__(**kwargs)
        self.U = tf.constant(U_dict, dtype=tf.complex64)
        self.V = tf.constant(V_dict, dtype=tf.complex64)
        self.c = tf.constant(dict_c, dtype=tf.float32)
        self.c_complex = tf.complex(self.c, tf.constant(0.0, dtype=tf.float32))
        self.n_iters = n_iters
        self.G = U_dict.shape[1]

    def build(self, input_shape):
        self.steps = [self.add_weight(name=f'step_{k}', shape=(), dtype=tf.float32,
                                       initializer=tf.keras.initializers.Constant(0.06))
                      for k in range(self.n_iters)]
        self.thresholds = [self.add_weight(name=f'thresh_{k}', shape=(), dtype=tf.float32,
                                            initializer=tf.keras.initializers.Constant(0.015))
                            for k in range(self.n_iters)]

    def call(self, y_real_imag):
        Y = tf.complex(y_real_imag[..., 0], y_real_imag[..., 1])
        batch = tf.shape(Y)[0]
        X = tf.zeros((batch, self.G, self.G), dtype=tf.float32)
        Uc = tf.math.conj(self.U)
        Vc = tf.math.conj(self.V)
        for k in range(self.n_iters):
            Xc = tf.cast(X, tf.complex64)
            Yhat = self.c_complex * tf.einsum('qi,bij,pj->bqp', self.U, Xc, self.V)
            R = Y - Yhat
            grad = self.c * tf.math.real(tf.einsum('qi,bqp,pj->bij', Uc, R, Vc))
            X = tf.nn.relu(X + self.steps[k] * grad - self.thresholds[k])
            X = tf.clip_by_value(X, 0.0, MAX_ISTA_VAL)   # <-- stability fix
        return tf.expand_dims(X, axis=-1)

print('LearnedISTA layer defined.')

### Part 5.1 — Ekta sample-e Learned-ISTA (untrained weight diye) ki dey dekho

In [ ]:
_probe_layer = LearnedISTA(U_dict, V_dict, DICT_C, n_iters=6)
_probe_out = _probe_layer(tf.constant(X_val_p16[:1]))
plt.figure(figsize=(4, 4))
plt.imshow(_probe_out[0, :, :, 0].numpy(), cmap='hot')
plt.title('Learned-ISTA output (untrained weights)\n-- ekhono kono training hoyni, tai eta shudhu physics-based coarse estimate')
plt.xticks([]); plt.yticks([]); plt.show()

## Part 6 — PIA-Net full architecture

**Architecture design note (keno hybrid?)**: prothom vabe shudhu Learned-ISTA-er upor pura bhorosha kore
model banano hoyechilo -- kintu extensive testing-e dekha gelo eta onek dhire converge kore (limited
data/epoch-e practically stuck thake), karon puro spatial-localization kaj-tar dায়িত্ব ekta hard-to-train
iterative layer-er upor। Tai final design-e ekta **parallel direct-learned CNN path** add kora hoyeche
(thik paper-er নিজের UNet baseline যেভাবে direct feature learning kore) -- eta model-take ekta doosra,
onek fast-converging route dey, ISTA branch converge korte na parle-o shomossha hobe na।

Architecture-er dui-ta path:
- **Path A (physics-informed)**: Learned-ISTA -> (32,32,1) sparse code map -- interpretable, physics-based।
- **Path B (direct-learned)**: raw observation-er upor shorashori small CNN, upsample kore (32,32,32) feature
  -- kono physics assumption nei, purapuri data-driven, robust fallback।

Duita path concat kore attention-refine + convolutional decoder diye 256x256 heatmap-e upsample kora hoy,
sesh-e ekta "physics skip connection" (ISTA output-take shorashori final layer porjonto niye asha) diye
guarantee kora hoy je physics prior kokhono completely hariye na jay।

In [ ]:
def attention_refine_block(x, d_model=32, n_heads=4):
    shape = x.shape[1:3]
    h = L.Conv2D(d_model, 1, padding='same')(x)
    seq = L.Reshape((shape[0] * shape[1], d_model))(h)
    attn_out = L.MultiHeadAttention(num_heads=n_heads, key_dim=d_model // n_heads)(seq, seq)
    seq = L.Add()([seq, attn_out])
    seq = L.LayerNormalization()(seq)
    h = L.Reshape((shape[0], shape[1], d_model))(seq)
    h = L.Conv2D(d_model, 1, padding='same', activation='relu')(h)
    return L.Add()([x, h])

def conv_block(x, filters):
    return L.Conv2D(filters, 3, padding='same', activation='relu')(x)

def build_pia_net(g_grid=G_GRID, m_out=M_OUT, n_ista_iters=6):
    inputs = tf.keras.Input(shape=(P_CB, Q_CB, 2), name='raw_observation')

    # ---- Path A: physics-informed sparse dictionary ----
    ista_map_raw = LearnedISTA(U_dict, V_dict, DICT_C, n_iters=n_ista_iters, name='learned_ista')(inputs)
    ista_map = L.Rescaling(1.0 / MAX_ISTA_VAL)(ista_map_raw)

    # ---- Path B: direct learned CNN encoder ----
    b = L.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    b = L.Conv2D(32, 3, padding='same', activation='relu')(b)
    b = L.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(b)  # 16x16 -> 32x32
    b = L.Conv2D(32, 3, padding='same', activation='relu')(b)

    fused = L.Concatenate()([ista_map, b])
    fused = conv_block(fused, 48)
    x = attention_refine_block(fused, d_model=48, n_heads=4)
    x = conv_block(x, 64)
    x = conv_block(x, 64)

    upsample_steps = int(np.log2(m_out // g_grid))
    filt = 64
    for _ in range(upsample_steps):
        x = L.Conv2DTranspose(filt, 3, strides=2, padding='same', activation='relu')(x)
        x = conv_block(x, filt)
        filt = max(filt // 2, 24)

    physics_skip = L.Resizing(m_out, m_out, interpolation='bilinear')(ista_map)
    x = L.Concatenate()([x, physics_skip])
    x = conv_block(x, 32)
    # target normalized to [0,1] per-sample (Part 7) -> sigmoid head. Bias-init close to the true
    # (very sparse, ~0.5%% foreground) base rate so training doesn't start from an uninformative flat 0.5.
    outputs = L.Conv2D(1, 3, padding='same', activation='sigmoid', name='heatmap',
                        kernel_initializer=tf.keras.initializers.RandomNormal(stddev=5e-3),
                        bias_initializer=tf.keras.initializers.Constant(-3.0))(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name='PIA-Net')

pia_net = build_pia_net()
pia_net.summary()
PIA_PARAM_COUNT = pia_net.count_params()
print('PIA-Net total params:', f'{PIA_PARAM_COUNT:,}')

## Part 7 — Loss function: normalized target + foreground-weighted MSE

**Key fix**: ground-truth heatmap-er raw amplitude (0 theke 90+) match korte chawa unnecessary-i kothin --
karon evaluation metric (Part 14) shudhu peak *location* dekhe, amplitude na (`cv2.normalize` diye always
rescale kore neya hoy)। Tai training-er age protita sample-er ground-truth ke নিজের max দিয়ে ভাগ kore
[0,1]-e normalize kora hocche -- model-ke ekhon shudhu shape/location shikhte hoy।

Extreme class-imbalance (~0.5%% pixel-i "foreground") handle korte weighted-MSE use kora hoy, jekhane
foreground pixel-er weight onek beshi (`alpha`).

In [ ]:
def normalize_gt_batch(Y):
    m = Y.reshape(len(Y), -1).max(axis=1).reshape(-1, 1, 1, 1)
    m = np.maximum(m, 1e-6)
    return Y / m

def weighted_mse(alpha=20.0):
    def loss_fn(y_true, y_pred):
        w = 1.0 + alpha * y_true
        return tf.reduce_mean(w * tf.square(y_pred - y_true))
    return loss_fn

Y_val_p16_norm = normalize_gt_batch(Y_val_p16)
print('Normalized target: min/max/mean =', Y_val_p16_norm.min(), Y_val_p16_norm.max(), Y_val_p16_norm.mean())

## Part 8 — Train/val split + compile

In [ ]:
X_pia_train, X_pia_val, Y_pia_train, Y_pia_val = train_test_split(
    X_val_p16, Y_val_p16_norm, test_size=0.15, random_state=42,
)
print('X_pia_train:', X_pia_train.shape, '  X_pia_val:', X_pia_val.shape)

# clipnorm=5.0: loose safety net -- age (clipnorm=1.0) khub tight ekta clip diye legitimate gradient
# signal-o throttle kore ফেলছিল (loss 5+ epoch dhore ekdom stuck thakto)। ISTA layer-er internal value-clip
# (Part 5) already runaway-divergence protect kore, tai outer clip ekhon shudhu loose safety net hishebe rakha.
pia_net.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1.5e-3, clipnorm=5.0),
    loss=weighted_mse(alpha=20.0),
)
print('Compiled.')

## Part 9 — Training

`EPOCHS` ekhane boro rakha hoyeche (tomar real Kaggle dataset onek boro, ei chhoto value paper-er small
dev-set proxy-r jonno na) -- `EarlyStopping` nijei best-point-e থামিয়ে দেবে (patience=15), overfitting
theke bachate। Jodi tomar GPU/time limited thake, `EPOCHS` kমিয়ে dite paro।

In [ ]:
EPOCHS = 80          # <-- boro rakha hoyeche, EarlyStopping nijei thamiye dibe
BATCH_SIZE = 16

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, min_delta=0.0003),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5, verbose=1),
    tf.keras.callbacks.TerminateOnNaN(),
]

history = pia_net.fit(
    X_pia_train, Y_pia_train,
    validation_data=(X_pia_val, Y_pia_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)
print(f'Training shesh -- {len(history.history["loss"])} actual epoch(s) cholechilo.')

### Part 9.1 — Training curve

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epoch'); plt.ylabel('Weighted MSE (normalized target)')
plt.title('PIA-Net training curve'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Part 10 — Inference utility functions (blob detection, angle conversion, metrics)

Paper-er `TVT_Blob_Inference.py`-r shathe identical logic -- model-er heatmap output theke peak (blob)
detect kore, angle-e convert kore, ground-truth-er shathe Hungarian-match kore, RMSE/Pd hishab kore।

In [ ]:
def prepare_prediction_for_peaks(prediction):
    prediction_map = prediction[:, :, 0].numpy()
    img_norm = cv2.normalize(prediction_map, None, 0, 255, cv2.NORM_MINMAX)
    return img_norm.astype(np.uint8)

def get_blob_detector():
    params = cv2.SimpleBlobDetector_Params()
    params.filterByColor = True
    params.blobColor = 255
    params.minThreshold = 0
    params.maxThreshold = 255
    params.filterByArea = True
    params.minArea = 1
    params.maxArea = 1000
    params.filterByCircularity = False
    params.filterByConvexity = False
    params.filterByInertia = False
    return cv2.SimpleBlobDetector_create(params)

detector = get_blob_detector()

def reorder_keypoints(keypoints, img_norm):
    coords = np.array([kp.pt for kp in keypoints])
    if len(coords) == 0:
        return [], np.array([])
    coords_rounded = np.round(coords).astype(int)
    amplitudes = []
    for (x, y) in coords_rounded:
        if 0 <= y < img_norm.shape[0] and 0 <= x < img_norm.shape[1]:
            amplitudes.append(img_norm[y, x])
        else:
            amplitudes.append(0)
    amplitudes = np.array(amplitudes)
    order = np.argsort(-amplitudes)
    return [keypoints[i] for i in order], amplitudes[order]

def get_blob_peaks(pred, detector):
    img_norm = prepare_prediction_for_peaks(pred)
    keypoints = detector.detect(img_norm)
    keypoints, amplitudes = reorder_keypoints(keypoints, img_norm)
    peaks = np.array([kp.pt for kp in keypoints]) if len(keypoints) > 0 else np.zeros((0, 2))
    return peaks, amplitudes

In [ ]:
def wrap_2pi_to_minus_pi(a):
    a = np.asarray(a)
    return np.where(a > np.pi, a - 2 * np.pi, a)

def peaks_to_angles(peaks, margin_factor=3.0, sigma=0.07, grid_size=256):
    if peaks.shape[0] == 0:
        return np.array([]), np.array([])
    margin = margin_factor * sigma
    peaks_x_y = peaks.T
    extended_range = 2 * np.pi + 2 * margin
    freqs_ext = -margin + (peaks_x_y / grid_size) * extended_range
    freqs_minus_pi = wrap_2pi_to_minus_pi(freqs_ext)
    psi_est = np.arccos(-freqs_minus_pi[1] / np.pi)
    phi_est = np.arccos(freqs_minus_pi[0] / np.pi)
    return psi_est, phi_est

def permute_pairs(A, B):
    A = np.asarray(A); B = np.asarray(B)
    dist_matrix = np.linalg.norm(A[:, np.newaxis, :] - B[np.newaxis, :, :], axis=2)
    row_ind, col_ind = linear_sum_assignment(dist_matrix)
    return [(tuple(A[i]), tuple(B[j])) for i, j in zip(row_ind, col_ind)]

def prepare_for_metric(angles_est, feat):
    Lp = feat.shape[-1]
    if len(angles_est[0]) < Lp:
        psi_est = np.full((Lp,), np.nan)
        phi_est = np.full((Lp,), np.nan)
        psi_true, phi_true = feat[0], feat[1]
    else:
        angles_est = (angles_est[0][:Lp], angles_est[1][:Lp])
        pairs_est = list(zip(angles_est[0], angles_est[1]))
        pairs_true = list(zip(feat[0], feat[1]))
        permuted = permute_pairs(pairs_true, pairs_est)
        first = [p[0] for p in permuted]
        second = [p[1] for p in permuted]
        psi_true, phi_true = zip(*first)
        psi_est, phi_est = zip(*second)
    return np.array([psi_true, phi_true]), np.array([psi_est, phi_est])

def get_ang_difference(gt_angles, pred_angles):
    H = np.angle(np.exp(1j * gt_angles) * np.exp(-1j * pred_angles))
    return (H * (180 / np.pi)).flatten()

def filter_angles(ang_dif_flat, max_deg_error=1.0):
    good = ang_dif_flat[np.abs(ang_dif_flat) <= max_deg_error]
    bad = ang_dif_flat[np.abs(ang_dif_flat) > max_deg_error]
    return good, bad

## Part 11 — Ekta single test example diye pura inference stage-by-stage dekho

In [ ]:
def run_single_example(model, X, Y_norm, feat, idx, sigma=0.07, grid_size=256):
    data = X[idx]
    gt = Y_norm[idx]
    pred = model(tf.expand_dims(data, axis=0), training=False)
    pred = tf.squeeze(pred, axis=0)

    peaks, amps = get_blob_peaks(pred, detector)
    Lp = feat.shape[-1] if feat is not None else len(peaks)
    order = np.argsort(-amps)
    peaks_sorted = peaks[order[:Lp]] if len(peaks) > 0 else peaks
    angles_est = peaks_to_angles(peaks_sorted, sigma=sigma, grid_size=grid_size)

    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    axs[0].imshow(data[:, :, 0], cmap='viridis'); axs[0].set_title('Input (real part, 16x16)')
    axs[1].imshow(gt[:, :, 0], cmap='hot'); axs[1].set_title('Ground truth (normalized)')
    axs[2].imshow(pred.numpy()[:, :, 0], cmap='hot'); axs[2].set_title('PIA-Net prediction')
    if len(peaks_sorted) > 0:
        axs[2].scatter(peaks_sorted[:, 0], peaks_sorted[:, 1], c='cyan', marker='x', s=60)
    for ax in axs:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()
    return angles_est

if X_test_p16 is not None:
    Y_test_norm = normalize_gt_batch(Y_test)
    _ = run_single_example(pia_net, X_test_p16, Y_test_norm, feat_test[0], idx=0)
    _ = run_single_example(pia_net, X_test_p16, Y_test_norm, feat_test[6], idx=6)
else:
    print('test set nei -- Part 11 skip.')

## Part 12 — Full evaluation: shob SNR condition-e RMSE / Pd hishab koro

Paper-er evaluation protocol-i follow kora hocche: proti (L=3, SNR, P=16) condition-er jonno RMSE
(1-degree-er modhdhe match kora angle-gulor upor) ar Pd (detection probability)।

In [ ]:
def evaluate_pia_net(model, X, meta, feat, max_samples=None):
    n = len(X) if max_samples is None else min(len(X), max_samples)
    results = {}
    for i in range(n):
        data = X[i]
        Lp, SNR, QP = int(meta[i, 0]), int(meta[i, 1]), int(meta[i, 2])
        cond = (Lp, SNR, QP)
        pred = model(tf.expand_dims(data, axis=0), training=False)
        pred = tf.squeeze(pred, axis=0)
        peaks, amps = get_blob_peaks(pred, detector)
        order = np.argsort(-amps)
        peaks = peaks[order[:Lp]] if len(peaks) else peaks
        angles_est = peaks_to_angles(peaks, sigma=0.07, grid_size=Y_test.shape[1])
        gt_a, pred_a = prepare_for_metric(angles_est, feat[i])
        results.setdefault(cond, []).append({'gt': gt_a, 'pred': pred_a})

    rmse, pd_ = {}, {}
    for cond, examples in results.items():
        good_all, bad_all = [], []
        for ex in examples:
            if np.isnan(ex['pred']).any():
                bad_all.append(np.full(ex['gt'].size, 999.0))
                continue
            diffs = get_ang_difference(ex['gt'], ex['pred'])
            good, bad = filter_angles(diffs, max_deg_error=1.0)
            good_all.append(good); bad_all.append(bad)
        good_all = np.concatenate(good_all) if good_all else np.array([])
        bad_all = np.concatenate(bad_all) if bad_all else np.array([])
        total = len(good_all) + len(bad_all)
        rmse[cond] = np.sqrt(np.mean(good_all ** 2)) if len(good_all) else np.nan
        pd_[cond] = len(good_all) / total if total else np.nan
    return rmse, pd_

if X_test_p16 is not None:
    pia_rmse, pia_pd = evaluate_pia_net(pia_net, X_test_p16, meta_test, feat_test)
    for cond in sorted(pia_rmse, key=lambda c: c[1]):
        print(cond, f'RMSE={pia_rmse[cond]:.4f}', f'Pd={pia_pd[cond]:.4f}')
else:
    pia_rmse, pia_pd = {}, {}
    print('test set nei -- evaluation skip.')

## Part 13 — Baseline reference: UNet / ResNet (paper-er repo-r own verified reproduction)

**Eগুলো hardcoded, protibar notebook run korle abar UNet/ResNet train hobe na** (oita heavy -- ResNet-e
64-ta stacked residual block, UNet-e 31M+ parameter, train korte GPU-te ghonta-r por ghonta lagে)। Ei
number-gulo ei repo-r committed reference file (`DL_DOA/figures_unet/*.pkl`, `DL_DOA/figures_resnet/*.pkl`)
theke neya -- mane eগুলো **actually-run, verified result**, paper-er নিজের claim na (jodi-o kache-kachi)।

In [ ]:
UNET_REF_RMSE = {
    (3,-10,16): 0.5498, (3,-5,16): 0.5069, (3,0,16): 0.4548, (3,5,16): 0.3777,
    (3,10,16): 0.3047, (3,15,16): 0.2556, (3,20,16): 0.2297, (3,25,16): 0.2086,
}
UNET_REF_PD = {
    (3,-10,16): 0.2226, (3,-5,16): 0.4706, (3,0,16): 0.6788, (3,5,16): 0.8127,
    (3,10,16): 0.8883, (3,15,16): 0.9244, (3,20,16): 0.9439, (3,25,16): 0.9509,
}
RESNET_REF_RMSE = {
    (3,-10,16): 0.5532, (3,-5,16): 0.5118, (3,0,16): 0.4581, (3,5,16): 0.3920,
    (3,10,16): 0.3256, (3,15,16): 0.2792, (3,20,16): 0.2528, (3,25,16): 0.2377,
}
RESNET_REF_PD = {
    (3,-10,16): 0.2043, (3,-5,16): 0.4366, (3,0,16): 0.6396, (3,5,16): 0.7790,
    (3,10,16): 0.8623, (3,15,16): 0.8987, (3,20,16): 0.9250, (3,25,16): 0.9378,
}
UNET_PARAMS = 31_276_481
RESNET_PARAMS = 469_393

print('Reference table loaded (UNet & ResNet, paper Figs. 5-6 condition L=3, P=16).')

## Part 14 — PIA-Net vs UNet vs ResNet: RMSE, Pd, ebong model size comparison

In [ ]:
snrs = sorted(set(k[1] for k in UNET_REF_RMSE))

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))

axs[0].plot(snrs, [UNET_REF_RMSE[(3,s,16)] for s in snrs], 'o-', label='UNet (reference)')
axs[0].plot(snrs, [RESNET_REF_RMSE[(3,s,16)] for s in snrs], 's-', label='ResNet (reference)')
if pia_rmse:
    pia_snrs = sorted(set(k[1] for k in pia_rmse if k[0] == 3 and k[2] == 16))
    axs[0].plot(pia_snrs, [pia_rmse[(3,s,16)] for s in pia_snrs], '^-', label='PIA-Net (this run)', color='crimson')
axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('RMSE (deg)'); axs[0].set_title('RMSE vs SNR')
axs[0].legend(); axs[0].grid(alpha=0.3)

axs[1].plot(snrs, [UNET_REF_PD[(3,s,16)] for s in snrs], 'o-', label='UNet (reference)')
axs[1].plot(snrs, [RESNET_REF_PD[(3,s,16)] for s in snrs], 's-', label='ResNet (reference)')
if pia_pd:
    pia_snrs = sorted(set(k[1] for k in pia_pd if k[0] == 3 and k[2] == 16))
    axs[1].plot(pia_snrs, [pia_pd[(3,s,16)] for s in pia_snrs], '^-', label='PIA-Net (this run)', color='crimson')
axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('Pd'); axs[1].set_ylim(-0.02, 1.02)
axs[1].set_title('Detection probability vs SNR')
axs[1].legend(); axs[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
names = ['UNet', 'ResNet', 'PIA-Net']
params = [UNET_PARAMS, RESNET_PARAMS, PIA_PARAM_COUNT]
colors = ['#4c72b0', '#dd8452', '#c44e52']
bars = ax.bar(names, params, color=colors)
ax.set_yscale('log')
ax.set_ylabel('Parameter count (log scale)')
ax.set_title('Model size comparison')
for b, p in zip(bars, params):
    ax.text(b.get_x() + b.get_width()/2, p, f'{p:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

print(f'PIA-Net is {UNET_PARAMS/PIA_PARAM_COUNT:.1f}x smaller than UNet, '
      f'{RESNET_PARAMS/PIA_PARAM_COUNT:.2f}x the size of ResNet.')

## Part 15 — Honest verdict (auto-generated real number theke, kono hardcode/overclaim na)

Ei cell **actual computed PIA-Net result** o hardcoded reference compare kore ekta honest summary print
kore -- jodi gap boro thake, seta lukiye rakha hoyna, karon ei number-i tumi supervisor-ke show korbe।

In [ ]:
if pia_rmse:
    common_snrs = sorted(set(k[1] for k in pia_pd if k[0] == 3 and k[2] == 16))
    print('='*72)
    print('PIA-Net vs reference -- condition-by-condition')
    print('='*72)
    gaps_pd = []
    for s in common_snrs:
        cond = (3, s, 16)
        p_rmse, p_pd = pia_rmse.get(cond, float('nan')), pia_pd.get(cond, float('nan'))
        u_pd = UNET_REF_PD[cond]
        gap = u_pd - p_pd if not np.isnan(p_pd) else np.nan
        gaps_pd.append(gap)
        print(f'SNR={s:4d} dB | PIA-Net RMSE={p_rmse:.3f} Pd={p_pd:.3f}  '
              f'| UNet-ref Pd={u_pd:.3f}  | gap={gap:+.3f}')

    mean_gap = np.nanmean(gaps_pd)
    print()
    print('='*72)
    if mean_gap < 0.05:
        print(f'VERDICT: PIA-Net Pd is within ~{mean_gap:.2f} of the UNet reference on average -- '
              f'kache-kachi (kasakasi) result, {UNET_PARAMS/PIA_PARAM_COUNT:.0f}x kom parameter diye.')
    elif mean_gap < 0.25:
        print(f'VERDICT: PIA-Net-er average Pd gap UNet reference theke ~{mean_gap:.2f} -- '
              'moderate gap ache, kintu ei run **onek chhoto dataset-e** (500-er kom P=16 sample) hoyeche. '
              'Tomar full-size real Kaggle dataset-e (onek beshi sample + comparable epoch) train korle '
              'eta আরো kace ashar kotha, kintu eta ei notebook guarantee kore na.')
    else:
        print(f'VERDICT: PIA-Net-er average Pd gap UNet reference theke ~{mean_gap:.2f} -- eখনো boro gap ache. '
              'Ei chhoto test-e dekha gেছে model real spatial structure শিখছে (Part 11-er visualization-e '
              'dekho predicted peak ground-truth-er kache আসতে পারে), kintu paper-level accuracy-r jonno '
              'aro training data / epoch / hyperparameter tuning lagbe. Honest recommendation: eি number-o '
              'supervisor-ke show koro, shathe bolo je eta ekta lightweight physics-informed baseline-er '
              'প্রথম working iteration -- collapse/NaN bug fix kore real learning kora geche, kintu tuning '
              'still in-progress.')
    print('='*72)
else:
    print('test set na thakle final verdict compute kora jabe na.')